In [9]:
import pandas as pd
from pathlib import Path

# Diretório principal do projeto
PROJECT_ROOT = Path.cwd().parent

# Origem: arquivos Parquet da camada Silver
SOURCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "open_meteo"
    / "archive"
)

# Destino: arquivos Parquet da camada Gold
TARGET_DIR = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "open_meteo"
    / "archive"
)

print("Diretório do projeto:", PROJECT_ROOT)
print("Origem Silver:", SOURCE_DIR)
print("Destino Gold:", TARGET_DIR)

print("\nSilver existe?", SOURCE_DIR.exists())
print("Gold existe?", TARGET_DIR.exists())

Diretório do projeto: /home/mariarita-resende/Projetos/open_meteo_ingestion
Origem Silver: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/archive
Destino Gold: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/gold/open_meteo/archive

Silver existe? True
Gold existe? True


In [10]:
def get_parquet_files(directory: Path) -> list[Path]:
    if not directory.exists():
        raise FileNotFoundError(
            f"Diretório Silver não encontrado: {directory}"
        )

    return sorted(directory.rglob("*.parquet"))


parquet_files = get_parquet_files(SOURCE_DIR)

print(f"Total de arquivos Parquet encontrados: {len(parquet_files)}")

for path in parquet_files[:5]:
    print(path)

Total de arquivos Parquet encontrados: 68
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/archive/20260423/amsterdam_20260423.parquet
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/archive/20260423/barcelona_20260423.parquet
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/archive/20260423/berlin_20260423.parquet
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/archive/20260423/brussels_20260423.parquet
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/archive/20260423/budapest_20260423.parquet


In [11]:
sample_path = parquet_files[0]

sample_df = pd.read_parquet(sample_path, engine="pyarrow")

print("Arquivo analisado:", sample_path.name)
print("Quantidade de linhas:", len(sample_df))
print("Quantidade de colunas:", len(sample_df.columns))

print("\nColunas:")
print(sample_df.columns.tolist())

print("\nTipos de dados:")
print(sample_df.dtypes)

print("\nPrimeiras 5 linhas:")
display(sample_df.head())

Arquivo analisado: amsterdam_20260423.parquet
Quantidade de linhas: 168
Quantidade de colunas: 16

Colunas:
['source_name', 'source_type', 'city_name', 'ingestion_date', 'observation_time', 'temperature_celsius', 'feels_like_celsius', 'relative_humidity_pct', 'dew_point_2m', 'precipitation_mm', 'rain', 'snowfall', 'snow_depth', 'weather_code', 'wind_speed_10m', 'wind_direction_10m']

Tipos de dados:
source_name               object
source_type               object
city_name                 object
ingestion_date            object
observation_time          object
temperature_celsius      float64
feels_like_celsius       float64
relative_humidity_pct      int64
dew_point_2m             float64
precipitation_mm         float64
rain                     float64
snowfall                 float64
snow_depth               float64
weather_code               int64
wind_speed_10m           float64
wind_direction_10m         int64
dtype: object

Primeiras 5 linhas:


,source_name,source_type,city_name,ingestion_date,observation_time,temperature_celsius,feels_like_celsius,relative_humidity_pct,dew_point_2m,precipitation_mm,rain,snowfall,snow_depth,weather_code,wind_speed_10m,wind_direction_10m
0,open_meteo,archive,amsterdam,20260423,2026-04-23T00:00Z,8.8,6.1,80,5.5,0.0,0.0,0.0,0.0,0,10.8,20
1,open_meteo,archive,amsterdam,20260423,2026-04-23T01:00Z,8.7,6.1,78,5.0,0.0,0.0,0.0,0.0,0,9.4,36
2,open_meteo,archive,amsterdam,20260423,2026-04-23T02:00Z,8.4,5.6,81,5.3,0.0,0.0,0.0,0.0,0,11.2,33
3,open_meteo,archive,amsterdam,20260423,2026-04-23T03:00Z,8.0,5.3,82,5.1,0.0,0.0,0.0,0.0,0,10.4,28
4,open_meteo,archive,amsterdam,20260423,2026-04-23T04:00Z,8.0,5.4,82,5.1,0.0,0.0,0.0,0.0,0,9.4,14


In [12]:
print("Valores ausentes por coluna:")
print(sample_df.isna().sum())

print("\nTotal de valores ausentes:")
print(sample_df.isna().sum().sum())

print("\nPercentual de valores ausentes por coluna:")
print((sample_df.isna().mean() * 100).round(2))

Valores ausentes por coluna:
source_name              0
source_type              0
city_name                0
ingestion_date           0
observation_time         0
temperature_celsius      0
feels_like_celsius       0
relative_humidity_pct    0
dew_point_2m             0
precipitation_mm         0
rain                     0
snowfall                 0
snow_depth               0
weather_code             0
wind_speed_10m           0
wind_direction_10m       0
dtype: int64

Total de valores ausentes:
0

Percentual de valores ausentes por coluna:
source_name              0.0
source_type              0.0
city_name                0.0
ingestion_date           0.0
observation_time         0.0
temperature_celsius      0.0
feels_like_celsius       0.0
relative_humidity_pct    0.0
dew_point_2m             0.0
precipitation_mm         0.0
rain                     0.0
snowfall                 0.0
snow_depth               0.0
weather_code             0.0
wind_speed_10m           0.0
wind_direction_10

In [13]:
def build_daily_archive_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Extrair a data do horário da observação
    df["observation_date"] = pd.to_datetime(
        df["observation_time"],
        utc=True
    ).dt.strftime("%Y-%m-%d")

    # Definir as agregações diárias
    aggregations = {
        "temperature_celsius": ["mean", "min", "max"],
        "feels_like_celsius": "mean",
        "relative_humidity_pct": "mean",
        "dew_point_2m": "mean",
        "precipitation_mm": "sum",
        "rain": "sum",
        "snowfall": "sum",
        "wind_speed_10m": "mean",
    }

    daily_df = (
        df.groupby(
            [
                "source_name",
                "source_type",
                "city_name",
                "ingestion_date",
                "observation_date",
            ],
            as_index=False,
        )
        .agg(aggregations)
    )

    # Simplificar os nomes das colunas resultantes
    daily_df.columns = [
        "_".join(column).strip("_")
        if isinstance(column, tuple)
        else column
        for column in daily_df.columns
    ]

    daily_df = daily_df.rename(
        columns={
            "temperature_celsius_mean": "temperature_mean_celsius",
            "temperature_celsius_min": "temperature_min_celsius",
            "temperature_celsius_max": "temperature_max_celsius",
            "feels_like_celsius_mean": "feels_like_mean_celsius",
            "relative_humidity_pct_mean": "relative_humidity_mean_pct",
            "dew_point_2m_mean": "dew_point_mean_celsius",
            "precipitation_mm_sum": "precipitation_total_mm",
            "rain_sum": "rain_total_mm",
            "snowfall_sum": "snowfall_total_cm",
            "wind_speed_10m_mean": "wind_speed_mean_kmh",
        }
    )

    return daily_df

In [14]:
daily_sample_df = build_daily_archive_dataframe(sample_df)

print("Arquivo de origem:", sample_path.name)
print("Linhas horárias:", len(sample_df))
print("Linhas diárias:", len(daily_sample_df))
print("Quantidade de colunas Gold:", len(daily_sample_df.columns))

print("\nColunas:")
print(daily_sample_df.columns.tolist())

print("\nResultado da agregação diária:")
display(daily_sample_df)

Arquivo de origem: amsterdam_20260423.parquet
Linhas horárias: 168
Linhas diárias: 7
Quantidade de colunas Gold: 15

Colunas:
['source_name', 'source_type', 'city_name', 'ingestion_date', 'observation_date', 'temperature_mean_celsius', 'temperature_min_celsius', 'temperature_max_celsius', 'feels_like_mean_celsius', 'relative_humidity_mean_pct', 'dew_point_mean_celsius', 'precipitation_total_mm', 'rain_total_mm', 'snowfall_total_cm', 'wind_speed_mean_kmh']

Resultado da agregação diária:


,source_name,source_type,city_name,ingestion_date,observation_date,temperature_mean_celsius,temperature_min_celsius,temperature_max_celsius,feels_like_mean_celsius,relative_humidity_mean_pct,dew_point_mean_celsius,precipitation_total_mm,rain_total_mm,snowfall_total_cm,wind_speed_mean_kmh
0,open_meteo,archive,amsterdam,20260423,2026-04-23,11.387500,7.6,16.8,8.629167,63.416667,4.058333,0.0,0.0,0.0,11.112500
1,open_meteo,archive,amsterdam,20260423,2026-04-24,11.062500,7.3,16.4,8.083333,67.083333,4.795833,0.0,0.0,0.0,13.445833
2,open_meteo,archive,amsterdam,20260423,2026-04-25,10.862500,7.4,15.6,8.062500,64.666667,3.983333,0.0,0.0,0.0,11.229167
3,open_meteo,archive,amsterdam,20260423,2026-04-26,8.362500,4.2,12.9,6.066667,73.958333,3.654167,0.0,0.0,0.0,6.941667
4,open_meteo,archive,amsterdam,20260423,2026-04-27,8.737500,3.8,13.5,6.979167,78.083333,4.720833,2.7,0.8,0.0,4.695833
5,open_meteo,archive,amsterdam,20260423,2026-04-28,8.620833,6.4,10.9,5.629167,80.666667,5.345833,1.2,0.4,0.0,12.700000
6,open_meteo,archive,amsterdam,20260423,2026-04-29,10.408333,5.9,14.1,6.583333,62.208333,3.004167,0.0,0.0,0.0,16.633333


In [15]:
processed_files = 0
total_daily_rows = 0
errors = []

for path in parquet_files:
    try:
        # Ler o arquivo Parquet da camada Silver
        df = pd.read_parquet(path, engine="pyarrow")

        # Calcular as métricas diárias
        daily_df = build_daily_archive_dataframe(df)

        # Manter a estrutura de diretórios da origem
        relative_path = path.relative_to(SOURCE_DIR)
        output_path = TARGET_DIR / relative_path

        # Garantir que o diretório de destino exista
        output_path.parent.mkdir(parents=True, exist_ok=True)

        # Salvar o resultado em Parquet
        daily_df.to_parquet(
            output_path,
            engine="pyarrow",
            index=False
        )

        processed_files += 1
        total_daily_rows += len(daily_df)

    except Exception as error:
        errors.append({
            "arquivo": str(path),
            "erro": str(error)
        })

print("Arquivos processados:", processed_files)
print("Total de linhas diárias geradas:", total_daily_rows)
print("Quantidade de erros:", len(errors))

if errors:
    print("\nDetalhes dos erros:")
    for error in errors:
        print(error)

Arquivos processados: 68
Total de linhas diárias geradas: 140
Quantidade de erros: 0


In [16]:
# Listar os arquivos Parquet gerados na camada Gold
gold_files = sorted(TARGET_DIR.rglob("*.parquet"))

print("Total de arquivos Parquet na Gold:", len(gold_files))

# Selecionar um arquivo para validação
gold_sample_path = gold_files[0]

gold_sample_df = pd.read_parquet(
    gold_sample_path,
    engine="pyarrow"
)

print("\nArquivo analisado:", gold_sample_path.name)
print("Quantidade de linhas:", len(gold_sample_df))
print("Quantidade de colunas:", len(gold_sample_df.columns))

print("\nValores ausentes por coluna:")
print(gold_sample_df.isna().sum())

print("\nDados gerados:")
display(gold_sample_df)

Total de arquivos Parquet na Gold: 68

Arquivo analisado: amsterdam_20260423.parquet
Quantidade de linhas: 7
Quantidade de colunas: 15

Valores ausentes por coluna:
source_name                   0
source_type                   0
city_name                     0
ingestion_date                0
observation_date              0
temperature_mean_celsius      0
temperature_min_celsius       0
temperature_max_celsius       0
feels_like_mean_celsius       0
relative_humidity_mean_pct    0
dew_point_mean_celsius        0
precipitation_total_mm        0
rain_total_mm                 0
snowfall_total_cm             0
wind_speed_mean_kmh           0
dtype: int64

Dados gerados:


,source_name,source_type,city_name,ingestion_date,observation_date,temperature_mean_celsius,temperature_min_celsius,temperature_max_celsius,feels_like_mean_celsius,relative_humidity_mean_pct,dew_point_mean_celsius,precipitation_total_mm,rain_total_mm,snowfall_total_cm,wind_speed_mean_kmh
0,open_meteo,archive,amsterdam,20260423,2026-04-23,11.387500,7.6,16.8,8.629167,63.416667,4.058333,0.0,0.0,0.0,11.112500
1,open_meteo,archive,amsterdam,20260423,2026-04-24,11.062500,7.3,16.4,8.083333,67.083333,4.795833,0.0,0.0,0.0,13.445833
2,open_meteo,archive,amsterdam,20260423,2026-04-25,10.862500,7.4,15.6,8.062500,64.666667,3.983333,0.0,0.0,0.0,11.229167
3,open_meteo,archive,amsterdam,20260423,2026-04-26,8.362500,4.2,12.9,6.066667,73.958333,3.654167,0.0,0.0,0.0,6.941667
4,open_meteo,archive,amsterdam,20260423,2026-04-27,8.737500,3.8,13.5,6.979167,78.083333,4.720833,2.7,0.8,0.0,4.695833
5,open_meteo,archive,amsterdam,20260423,2026-04-28,8.620833,6.4,10.9,5.629167,80.666667,5.345833,1.2,0.4,0.0,12.700000
6,open_meteo,archive,amsterdam,20260423,2026-04-29,10.408333,5.9,14.1,6.583333,62.208333,3.004167,0.0,0.0,0.0,16.633333
